In [1]:
from EC_GNN import Mobility_ECGNN
from gnn.ECC_GNN_V2 import MCC_GNN
from advancedgnn import Advanced_Mobility_GNN

In [4]:
import torch
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader


dataset = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/graphswitharea.pt', weights_only=False)
total_graphs = len(dataset)

#train test split
train_size = int(0.8 * total_graphs)
test_size = total_graphs - train_size

print(f"Total Graphs: {total_graphs} | Training on: {train_size} | Testing on: {test_size}")

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

Total Graphs: 150 | Training on: 120 | Testing on: 30


In [6]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

# 1. Initialize your model
#model = Mobility_ECGNN(node_features_dim=3, edge_features_dim=1, hidden_dim=64)
model = MCC_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)
#model = Advanced_Mobility_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) # Adam is the standard for GNNs
#criterion = nn.MSELoss() #MSE
criterion = nn.L1Loss()

# training_graphs = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/testingpyg.pt', weights_only=False)
# train_loader = DataLoader(training_graphs, batch_size=32, shuffle=True)

#training loop
epochs = 100


for epoch in range(epochs):
    total_loss = 0

    model.train()
    for batch_data in train_loader:

        optimizer.zero_grad()
        predictions =  model(batch_data)

        loss = criterion(predictions.squeeze(), batch_data.y)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    total_test_error = 0

    with torch.no_grad():
     for batch_data in test_loader:

        # Make the prediction (remember, use model(), not model.forward()!)
        predictions = model(batch_data)

        # Calculate the Mean Squared Error (MSE) for this batch
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

# 3. Calculate final average error
     avg_test_mse = total_test_error / len(test_loader)


    if epoch % 1 == 0:
        print(f"Epoch {epoch} | Average Training Loss (MSE): {avg_loss:.4f}")
        print(f"Epoch {epoch} | Average Testing Loss (MSE): {avg_test_mse:.4f}")

Epoch 0 | Average Training Loss (MSE): 0.3510
Epoch 0 | Average Testing Loss (MSE): 0.1560
Epoch 1 | Average Training Loss (MSE): 0.1410
Epoch 1 | Average Testing Loss (MSE): 0.0447
Epoch 2 | Average Training Loss (MSE): 0.0801
Epoch 2 | Average Testing Loss (MSE): 0.0411
Epoch 3 | Average Training Loss (MSE): 0.0672
Epoch 3 | Average Testing Loss (MSE): 0.0881
Epoch 4 | Average Training Loss (MSE): 0.0740
Epoch 4 | Average Testing Loss (MSE): 0.0295
Epoch 5 | Average Training Loss (MSE): 0.0583
Epoch 5 | Average Testing Loss (MSE): 0.0491
Epoch 6 | Average Training Loss (MSE): 0.0486
Epoch 6 | Average Testing Loss (MSE): 0.0298
Epoch 7 | Average Training Loss (MSE): 0.0532
Epoch 7 | Average Testing Loss (MSE): 0.0734
Epoch 8 | Average Training Loss (MSE): 0.0651
Epoch 8 | Average Testing Loss (MSE): 0.0403
Epoch 9 | Average Training Loss (MSE): 0.0535
Epoch 9 | Average Testing Loss (MSE): 0.0317
Epoch 10 | Average Training Loss (MSE): 0.0534
Epoch 10 | Average Testing Loss (MSE): 0.04

In [6]:
testing_graphs = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/testingpyg2.pt', weights_only=False)
test_loader = DataLoader(testing_graphs, batch_size=32, shuffle=True)

In [7]:
model.eval()

# We will track the total error across all test graphs
total_test_error = 0.0

# 2. Turn off Autograd
# We use torch.no_grad() because we are not updating weights.
# This makes the forward pass incredibly fast and uses almost zero memory.
with torch.no_grad():
    for batch_data in test_loader:

        # Make the prediction (remember, use model(), not model.forward()!)
        predictions = model(batch_data)

        # Calculate the Mean Squared Error (MSE) for this batch
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

# 3. Calculate final average error
avg_test_mse = total_test_error / len(test_loader)

print(f"Final Test MSE: {avg_test_mse:.6f}")

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [9541, 3] but got: [9541, 4].

In [ ]:
real_graph = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/real_graph.pt', weights_only=False)
real_loader = DataLoader([real_graph], batch_size=32, shuffle=True)

In [ ]:
model.eval()

# We will track the total error across all test graphs
total_test_error = 0.0

# 2. Turn off Autograd
# We use torch.no_grad() because we are not updating weights.
# This makes the forward pass incredibly fast and uses almost zero memory.
with torch.no_grad():
    for batch_data in real_loader:

        # Make the prediction (remember, use model(), not model.forward()!)
        predictions = model(batch_data)

        # Calculate the Mean Squared Error (MSE) for this batch
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

# 3. Calculate final average error
avg_test_mse = total_test_error / len(real_loader)

print(f"Final Test: {avg_test_mse:.6f}")